In [ ]:
# study1_run_all.ipynb -- Study 1, all stages in one notebook (EDA -> features ->
# baseline -> Era 1 ramp characterization), for anyone who just wants to open ONE
# file and run it end to end instead of hunting through 4 separate notebooks.
#
# The 4 original notebooks (01_eda.ipynb, 02_features.ipynb, 03_baseline.ipynb,
# 04_era1_ramp_characterization.ipynb) still exist and are still the source of
# truth for each individual stage -- this is a convenience wrapper, not a
# replacement. If you're only interested in one stage, open that notebook
# directly instead.
#
# CREDENTIALS: this cell reads your Kaggle username/key from Colab's Secrets
# manager (the key icon in the left sidebar), NOT from a hardcoded string in this
# cell. Add two secrets named KAGGLE_USERNAME and KAGGLE_KEY there (toggle
# "Notebook access" on for each) before running this cell. Never type your actual
# key into a notebook cell you plan to commit -- this repo is public, and
# anything typed into a committed cell is visible to everyone and stays in git
# history even after you remove it later.

!git clone https://github.com/HalcyonVector/Grid-Sentinel.git
%cd Grid-Sentinel/ML/Study1
!pip install lightgbm -q

import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!kaggle datasets download -d halcyonvector/india-power-grid-nldc-daily-psp-reports -p data --unzip


In [ ]:
# --- Stage 1: 01_eda.ipynb -- Study 1 EDA ---
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
print(df.shape)  # expect (2660, 144)
print(df["date"].min(), df["date"].max())

# --- Confirm target column ---
target_candidates = [
    "max_demand_met_total_mw",
    "evening_peak_demand_total_mw",
    "energy_met_total_mu",
]
for c in target_candidates:
    print(c, c in df.columns)

# --- Missingness ---
null_pct = df.isna().mean().sort_values(ascending=False) * 100
print(null_pct.head(20))

# --- Demand over time ---
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["max_demand_met_total_mw"])
plt.title("National max demand met (MW) over time")
plt.show()

# --- Weekly/yearly seasonality ---
df["dow"] = df["date"].dt.dayofweek
df["year"] = df["date"].dt.year
df.groupby("dow")["max_demand_met_total_mw"].mean().plot(kind="bar", title="Avg demand by day of week")
plt.show()

df.groupby("year")["max_demand_met_total_mw"].mean().plot(kind="bar", title="Avg demand by year")
plt.show()

# --- RES share trend (needed for Era 1 later) ---
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["share_res_pct"])
plt.title("RES share % over time")
plt.show()

# --- Known gaps: cross-check against the known-gaps list from Phase 0 ---
full_range = pd.date_range(df["date"].min(), df["date"].max())
missing_dates = full_range.difference(df["date"])
print(f"{len(missing_dates)} missing dates (expect ~69 known gaps -- see Pipeline/known_gaps.json)")


In [ ]:
# --- Stage 2: 02_features.ipynb -- feature engineering ---
import pandas as pd
from features import build_feature_table, TARGET

df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

feat_df = build_feature_table(df)

print(feat_df.shape)
print(feat_df.columns.tolist())


In [ ]:
# --- Stage 3: 03_baseline.ipynb -- LightGBM baseline ---
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error

from features import build_feature_table, make_next_day_target, TARGET

df = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
feat_df = build_feature_table(df)
feat_df = make_next_day_target(feat_df)  # TARGET -> tomorrow's actual demand; TARGET_today preserved

feature_cols = [c for c in feat_df.columns if
                c.startswith(f"{TARGET}_lag") or
                c.startswith(f"{TARGET}_roll") or
                c in ("dow", "month", "year", "is_weekend", "day_of_year")]

feat_df = feat_df.dropna(subset=[f"{TARGET}_lag365", TARGET]).reset_index(drop=True)
# --- Time-aware split ---
train = feat_df[feat_df["year"] <= 2022]
val = feat_df[feat_df["year"] == 2023]
test = feat_df[feat_df["year"] >= 2024]

print("train:", train.shape, "val:", val.shape, "test:", test.shape)

# NOTE: anchor is TARGET_today (today's actual demand), NOT TARGET_lag1.
# lag1 was computed relative to the pre-shift target, so after make_next_day_target()
# it's 2 days behind the new target, not 1 -- see features.py's make_next_day_target()
# docstring for the full story. Using TARGET_today here is what makes both the
# delta-target and the naive-persistence baseline below correctly anchored.
X_train, y_train = train[feature_cols], train[TARGET] - train[f"{TARGET}_today"]
X_val, y_val = val[feature_cols], val[TARGET] - val[f"{TARGET}_today"]
X_test, y_test = test[feature_cols], test[TARGET]

model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

preds_test = model.predict(X_test) + test[f"{TARGET}_today"].values

# --- Naive persistence baseline (today's actual value = tomorrow's prediction) ---
naive_preds = test[f"{TARGET}_today"]

def report(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{name}: MAPE={mean_absolute_percentage_error(y_true, y_pred):.4f} "
          f"RMSE={rmse:.1f} "
          f"MAE={mean_absolute_error(y_true, y_pred):.1f}")

report("LightGBM", y_test, preds_test)
report("Naive persistence", y_test, naive_preds)

# --- Feature importance ---
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importance.head(20))

# --- Forecast residual (needed by Phase 4) ---
test = test.copy()
test["forecast"] = preds_test
test["residual"] = test[TARGET] - test["forecast"]
test[["date", TARGET, "forecast", "residual"]].to_csv("data/study1_forecast_residual.csv", index=False)


In [ ]:
# --- Stage 4: 04_era1_ramp_characterization.ipynb -- Era 1 (2019-2022) intra-day ramp analysis ---
import pandas as pd
import matplotlib.pyplot as plt

hourly = pd.read_csv("data/study1_hourly.csv")
hourly["datetime"] = pd.to_datetime(hourly["datetime"], format="%Y-%m-%d %H:%M:%S")
hourly["date"] = hourly["datetime"].dt.normalize()
daily = pd.read_csv("data/study1_daily.csv", parse_dates=["date"])

# Restrict to Era 1
hourly = hourly[hourly["date"].dt.year.between(2019, 2022)].copy()
daily_era1 = daily[daily["date"].dt.year.between(2019, 2022)][["date", "share_res_pct"]]

DEMAND_COL = "National Hourly Demand"  # confirm exact column name in your CSV

# --- Hour-to-hour ramp (delta) per day ---
hourly = hourly.sort_values("datetime").reset_index(drop=True)  # sort by full datetime, not just date -- sort_values on "date" alone
# isn't guaranteed stable across the 24 same-date rows per day, which could scramble hour order
hourly["ramp"] = hourly[DEMAND_COL].diff()

# Ramp magnitude = daily max absolute hour-to-hour change
daily_ramp = hourly.groupby(hourly["date"].dt.date)["ramp"].apply(lambda x: x.abs().max())
daily_ramp = daily_ramp.reset_index()
daily_ramp.columns = ["date", "ramp_magnitude"]
daily_ramp["date"] = pd.to_datetime(daily_ramp["date"])

# Ramp frequency = count of hours where |delta| exceeds a threshold (e.g. top 10% of ramps)
threshold = hourly["ramp"].abs().quantile(0.9)
daily_freq = hourly.groupby(hourly["date"].dt.date)["ramp"].apply(lambda x: (x.abs() > threshold).sum())
daily_freq = daily_freq.reset_index()
daily_freq.columns = ["date", "ramp_frequency"]
daily_freq["date"] = pd.to_datetime(daily_freq["date"])

# --- Merge with RES share ---
merged = daily_ramp.merge(daily_freq, on="date").merge(daily_era1, on="date", how="left")

# --- Monthly aggregation for trend clarity ---
merged["month"] = merged["date"].dt.to_period("M")
monthly = merged.groupby("month").agg(
    ramp_magnitude=("ramp_magnitude", "mean"),
    ramp_frequency=("ramp_frequency", "mean"),
    share_res_pct=("share_res_pct", "mean"),
).reset_index()
monthly["month"] = monthly["month"].dt.to_timestamp()

# --- The key chart: ramp magnitude trend vs RES share ---
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(monthly["month"], monthly["ramp_magnitude"], color="tab:blue", label="Ramp magnitude (MW)")
ax1.set_ylabel("Ramp magnitude (MW)", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(monthly["month"], monthly["share_res_pct"], color="tab:orange", label="RES share %")
ax2.set_ylabel("RES share %", color="tab:orange")

plt.title("Era 1 (2019-2022): Intra-day ramp magnitude vs RES share")
plt.show()

# --- Correlation check -
print(monthly[["ramp_magnitude", "ramp_frequency", "share_res_pct"]].corr())

monthly.to_csv("data/era1_ramp_vs_res_share.csv", index=False)
